# Módulo 2 — Mini-Projeto: Monitoramento Estrutural de uma Passarela Instrumentada

**Substitui a lista de exercícios do Cap. 4.** Os mesmos conceitos de Álgebra Linear Numérica
(matriz de Vandermonde/interpolação, média móvel, mínimos quadrados, decomposição LU, matrizes
de projeção) agora aparecem dentro de um único projeto de engenharia, em vez de exercícios
isolados.

## Contexto

Você foi contratado(a) como engenheiro(a) júnior de uma equipe de instrumentação estrutural.
Uma passarela de pedestres foi equipada com sensores de deslocamento para monitoramento contínuo,
e sua equipe precisa: calibrar os sensores, filtrar o ruído dos sinais medidos, caracterizar a
rigidez da estrutura a partir de um ensaio de carga, simular a resposta a diferentes cenários de
carregamento de pedestres, e verificar a consistência de sensores redundantes instalados por
segurança.

## Como o projeto está organizado

Cada etapa tem duas partes:

- **NÚCLEO (todos):** o que todo aluno deve resolver, independentemente do semestre. Usa apenas
  as ferramentas já vistas em aula, com o roteiro bem guiado.
- **APROFUNDAMENTO (desafio):** uma extensão mais aberta da mesma etapa, recomendada especialmente
  para quem está em fase final do curso — exige comparar métodos, interpretar resultados com mais
  autonomia ou generalizar o código. Não é obrigatória para a nota mínima, mas conta como
  diferencial.

Como nas atividades anteriores, **o código não é fornecido** — apenas o enunciado e uma célula de
validação ao final de cada parte. Resolva as etapas em ordem (algumas reaproveitam variáveis de
etapas anteriores).

---
## Etapa 1 — Calibração do sensor de deslocamento

*(Conceito: matriz de Vandermonde e interpolação polinomial)*

O sensor não mede deslocamento diretamente — ele devolve uma tensão elétrica (V), que precisa ser
convertida em deslocamento (mm) por meio de uma curva de calibração. Em bancada, o sensor foi
posicionado em 6 deslocamentos de referência conhecidos, e a tensão correspondente foi registrada:

```
d_ref     (mm) = [ 0,      5,      10,     15,    20,    25   ]
V_leitura (V)  = [ 0.0,    1.0175, 2.07,   3.1575, 4.28, 5.4375 ]
```

### NÚCLEO
1. Construa a matriz de Vandermonde `V1` (6×6) a partir de `V_leitura` (use `np.vander`).
2. Resolva o sistema `V1 @ coef1 = d_ref` para obter os coeficientes do polinômio de calibração
   (guarde em `coef1`).
3. Aplique a calibração às novas leituras `V_novas = [0.55, 1.85, 3.65]` usando `np.polyval`,
   guardando o resultado em `d_estim` (deslocamento estimado, em mm, para cada leitura nova).

In [ ]:
# Etapa 1 — Núcleo
import numpy as np

d_ref = np.array([0., 5., 10., 15., 20., 25.])
V_leitura = np.array([0., 1.0175, 2.07, 3.1575, 4.28, 5.4375])
V_novas = np.array([0.55, 1.85, 3.65])

# Escreva aqui o código dos itens 1 a 3

V1 = np.vander(V_leitura, 6)
coef1 = np.linalg.solve(V1, d_ref)
d_estim = np.polyval(coef1, V_novas)

In [ ]:
# Validação — Etapa 1 Núcleo (não modifique esta célula)
import numpy as np

assert "V1" in dir() and "coef1" in dir() and "d_estim" in dir(), "Defina V1, coef1 e d_estim"

V1_esperado = np.vander(V_leitura, 6)
assert np.allclose(np.array(V1, dtype=float), V1_esperado), "V1 não confere com vander(V_leitura, 6)"

coef1_esperado = np.linalg.solve(V1_esperado, d_ref)
assert np.allclose(np.array(coef1, dtype=float), coef1_esperado, atol=1e-4), "coef1 não confere com a solução de V1 @ coef1 = d_ref"

d_estim_esperado = np.polyval(coef1_esperado, V_novas)
assert np.allclose(np.array(d_estim, dtype=float), d_estim_esperado, atol=1e-3), "d_estim não confere com o esperado"

print("Etapa 1 (Núcleo) OK! d_estim ≈", np.round(d_estim_esperado, 3), "mm")


### APROFUNDAMENTO — Etapa 1 (desafio opcional)

O intervalo calibrado vai de V = 0 até V = 5.4375 (os limites de `V_leitura`). Calcule
`d_extrapolado = np.polyval(coef1, 10.0)` — uma leitura de tensão bem fora do intervalo
calibrado. O valor calculado existe matematicamente (o polinômio pode ser avaliado em qualquer
ponto), mas isso não significa que ele seja confiável.

Na célula de texto abaixo, explique: por que um polinômio interpolador de grau 5, ajustado a 6
pontos dentro de um intervalo, não deveria ser usado para *extrapolar* (estimar fora desse
intervalo)? O que poderia acontecer com um conjunto de calibração diferente, menos "bem
comportado"?

In [ ]:
# Etapa 1 — Aprofundamento
d_extrapolado = np.polyval(coef1, 10.0)
print(d_extrapolado)

**Minha explicação:**

Um polinômio de grau 5 passa exatamente pelos 6 pontos, mas fora do intervalo calibrado nada o limita e ele diverge rapidamente (aqui, V = 10 V dá ≈ 43,4 mm, sem nenhuma garantia de confiabilidade). Com dados menos "bem comportados" ou com ruído, haveria oscilações (fenômeno de Runge) e a extrapolação seria ainda pior.

In [ ]:
# Validação — Etapa 1 Aprofundamento (não modifique esta célula)
import numpy as np

if "d_extrapolado" in dir():
    esperado = np.polyval(coef1_esperado if "coef1_esperado" in dir() else np.linalg.solve(np.vander(V_leitura,6), d_ref), 10.0)
    assert np.isclose(d_extrapolado, esperado, atol=1e-2), "d_extrapolado não confere"
    print("Etapa 1 (Aprofundamento) OK! d_extrapolado ≈", round(float(d_extrapolado), 3), "mm")
else:
    print("Etapa 1 (Aprofundamento) não resolvida — tudo bem, é opcional.")


---
## Etapa 2 — Filtragem de ruído do sinal

*(Conceito: média móvel, sem laços `for`)*

Durante um teste de travessia de pedestre, o sensor (já calibrado) registrou o seguinte sinal de
deslocamento ao longo do tempo, com ruído de medição:

```
sinal_bruto (mm) = [0.2, 1.8, 4.9, 8.1, 11.8, 14.9, 17.2, 18.9, 17.5, 15.1, 11.9, 8.0, 4.8, 1.9, 0.3]
```

### NÚCLEO
Aplique uma média móvel de janela 3 (como vista em aula) e guarde o resultado em
`media_movel3`. **Não use laços `for`** — resolva com fatiamento de arrays.

In [ ]:
# Etapa 2 — Núcleo
sinal_bruto = np.array([0.2, 1.8, 4.9, 8.1, 11.8, 14.9, 17.2, 18.9, 17.5, 15.1, 11.9, 8.0, 4.8, 1.9, 0.3])

# Escreva aqui o código para calcular media_movel3 (sem for)

media_movel3 = (sinal_bruto[:-2] + sinal_bruto[1:-1] + sinal_bruto[2:]) / 3

In [ ]:
# Validação — Etapa 2 Núcleo (não modifique esta célula)
import numpy as np

assert "media_movel3" in dir(), "Defina a variável media_movel3"
esperado3 = (sinal_bruto[:-2] + sinal_bruto[1:-1] + sinal_bruto[2:]) / 3
assert np.allclose(np.array(media_movel3, dtype=float), esperado3), "media_movel3 não confere"
assert len(np.array(media_movel3)) == len(sinal_bruto) - 2, "media_movel3 deveria ter len(sinal_bruto) - 2 elementos"

print("Etapa 2 (Núcleo) OK! Pico bruto:", sinal_bruto.max(), " Pico filtrado (janela 3):", round(media_movel3.max(), 3))


### APROFUNDAMENTO — Etapa 2 (desafio opcional)

Generalize sua solução para uma janela de tamanho 5 (`media_movel5`, ainda sem `for`). Compare o
pico (valor máximo) do sinal bruto, da média móvel de janela 3 e da média móvel de janela 5.

Na célula de texto, explique o compromisso (trade-off): por que uma janela maior filtra mais
ruído, mas também "achata" picos reais do sinal? Para um sinal de deslocamento estrutural, por
que isso é uma preocupação de engenharia (e não só uma questão matemática)?

In [ ]:
# Etapa 2 — Aprofundamento
media_movel5 = (sinal_bruto[:-4] + sinal_bruto[1:-3] + sinal_bruto[2:-2]
                + sinal_bruto[3:-1] + sinal_bruto[4:]) / 5
print(sinal_bruto.max(), media_movel3.max(), media_movel5.max())

**Minha explicação:**

Picos: bruto 18,9 mm; janela 3 → 17,87 mm; janela 5 → 16,72 mm. Janelas maiores promediam mais pontos, reduzindo o ruído, mas também rebaixam o pico real. Em engenharia isso importa: subestimar o deslocamento máximo pode levar a uma avaliação de segurança otimista demais.

In [ ]:
# Validação — Etapa 2 Aprofundamento (não modifique esta célula)
import numpy as np

if "media_movel5" in dir():
    esperado5 = (sinal_bruto[:-4] + sinal_bruto[1:-3] + sinal_bruto[2:-2] + sinal_bruto[3:-1] + sinal_bruto[4:]) / 5
    assert np.allclose(np.array(media_movel5, dtype=float), esperado5), "media_movel5 não confere"
    print("Etapa 2 (Aprofundamento) OK! Pico janela 5:", round(media_movel5.max(), 3))
else:
    print("Etapa 2 (Aprofundamento) não resolvida — tudo bem, é opcional.")


---
## Etapa 3 — Rigidez estrutural por mínimos quadrados

*(Conceito: mínimos quadrados — equações normais e `scipy.linalg.lstsq`)*

Em um ensaio estático, cargas conhecidas foram aplicadas no meio do vão e o deslocamento (já
calibrado e filtrado) foi registrado:

```
F (N)  = [0, 500, 1000, 1500, 2000, 2500]
d (mm) = [0.3, 10.6, 19.4, 30.7, 39.5, 50.6]
```

Assumindo comportamento aproximadamente elástico-linear (d ≈ a·F + b), a rigidez da estrutura é
k = 1/a (N/mm).

### NÚCLEO
1. Monte a matriz de projeto `A3` com colunas `[F, 1]`.
2. Resolva o ajuste `d ≈ A3 @ [a, b]` **pelas equações normais**
   ((A3ᵀA3)⁻¹A3ᵀd), guardando `[a, b]` em `coef3`.
3. Repita usando `scipy.linalg.lstsq(A3, d)`, guardando em `coef3_lstsq` — os dois resultados
   devem praticamente coincidir.
4. Calcule a rigidez estimada `k_estimado = 1 / a` (N/mm).

In [ ]:
# Etapa 3 — Núcleo
F = np.array([0., 500., 1000., 1500., 2000., 2500.])
d = np.array([0.3, 10.6, 19.4, 30.7, 39.5, 50.6])

# Escreva aqui o código dos itens 1 a 4

A3 = np.column_stack([F, np.ones_like(F)])
coef3 = np.linalg.inv(A3.T @ A3) @ A3.T @ d

import scipy.linalg as sl
coef3_lstsq = sl.lstsq(A3, d)[0]

k_estimado = 1 / coef3[0]

In [ ]:
# Validação — Etapa 3 Núcleo (não modifique esta célula)
import numpy as np
import scipy.linalg as sl

assert "A3" in dir() and "coef3" in dir() and "coef3_lstsq" in dir() and "k_estimado" in dir(), \
    "Defina A3, coef3, coef3_lstsq e k_estimado"

A3_esperado = np.column_stack([F, np.ones_like(F)])
assert np.allclose(np.array(A3, dtype=float), A3_esperado), "A3 não confere com as colunas [F, 1]"

coef3_esperado = np.linalg.inv(A3_esperado.T @ A3_esperado) @ A3_esperado.T @ d
assert np.allclose(np.array(coef3, dtype=float), coef3_esperado, atol=1e-3), "coef3 não confere com as equações normais"

coef3_lstsq_esperado = sl.lstsq(A3_esperado, d)[0]
assert np.allclose(np.array(coef3_lstsq, dtype=float), coef3_lstsq_esperado, atol=1e-3), "coef3_lstsq não confere com scipy.linalg.lstsq"
assert np.allclose(np.array(coef3, dtype=float), np.array(coef3_lstsq, dtype=float), atol=1e-3), "coef3 e coef3_lstsq deveriam ser (quase) iguais"

assert np.isclose(k_estimado, 1 / coef3_esperado[0], rtol=1e-3), "k_estimado não confere com 1/a"

print("Etapa 3 (Núcleo) OK! Rigidez estimada k ≈", round(float(k_estimado), 2), "N/mm")


### APROFUNDAMENTO — Etapa 3 (desafio opcional)

Repita o ajuste com um modelo **quadrático**, d ≈ a₂·F² + a₁·F + a₀ (matriz de projeto `A3_quad`
com colunas `[F², F, 1]`, coeficientes em `coef3_quad`, resolvidos por mínimos quadrados — pode
usar `np.linalg.pinv` ou `scipy.linalg.lstsq`).

Calcule o **R²** (coeficiente de determinação) dos dois ajustes — linear (`r2_linear`) e
quadrático (`r2_quad`) — usando:

$$R^2 = 1 - \frac{\sum (d_i - \hat d_i)^2}{\sum (d_i - \bar d)^2}$$

Na célula de texto, justifique: o modelo quadrático melhora o ajuste o suficiente para valer a
pena usá-lo em vez do linear? (Pense em parcimônia — o princípio de preferir o modelo mais
simples que explica os dados igualmente bem.)

In [ ]:
# Etapa 3 — Aprofundamento
A3_quad = np.column_stack([F**2, F, np.ones_like(F)])
coef3_quad = sl.lstsq(A3_quad, d)[0]

ss_tot = np.sum((d - d.mean())**2)
r2_linear = 1 - np.sum((d - A3 @ coef3)**2) / ss_tot
r2_quad = 1 - np.sum((d - A3_quad @ coef3_quad)**2) / ss_tot
print(r2_linear, r2_quad)

**Minha justificativa:**

R² linear = 0,9990 e quadrático = 0,9991: ganho desprezível (~0,001), e o termo quadrático é minúsculo. Pelo princípio da parcimônia, o modelo linear basta e ainda dá uma rigidez direta (k ≈ 50,07 N/mm).

In [ ]:
# Validação — Etapa 3 Aprofundamento (não modifique esta célula)
import numpy as np

if "A3_quad" in dir() and "coef3_quad" in dir() and "r2_linear" in dir() and "r2_quad" in dir():
    A3_quad_esperado = np.column_stack([F**2, F, np.ones_like(F)])
    assert np.allclose(np.array(A3_quad, dtype=float), A3_quad_esperado), "A3_quad não confere com as colunas [F^2, F, 1]"

    coef3_quad_esperado = np.linalg.pinv(A3_quad_esperado) @ d
    assert np.allclose(np.array(coef3_quad, dtype=float), coef3_quad_esperado, atol=1e-3), "coef3_quad não confere"

    ss_tot = np.sum((d - d.mean())**2)
    r2_linear_esperado = 1 - np.sum((d - A3_esperado @ coef3_esperado)**2) / ss_tot
    r2_quad_esperado = 1 - np.sum((d - A3_quad_esperado @ coef3_quad_esperado)**2) / ss_tot
    assert np.isclose(r2_linear, r2_linear_esperado, atol=1e-3), "r2_linear não confere"
    assert np.isclose(r2_quad, r2_quad_esperado, atol=1e-3), "r2_quad não confere"

    print(f"Etapa 3 (Aprofundamento) OK! R² linear = {r2_linear_esperado:.4f}   R² quadrático = {r2_quad_esperado:.4f}")
else:
    print("Etapa 3 (Aprofundamento) não resolvida — tudo bem, é opcional.")


---
## Etapa 4 — Simulação de cenários de carregamento

*(Conceito: decomposição LU, reaproveitada para vários vetores de carga)*

A passarela foi modelada com 4 nós livres e matriz de rigidez `K` (fixa). A equipe precisa
avaliar o deslocamento em três cenários de carregamento de pedestres distintos:

```
K = | 8  -3   0   0|
    |-3   9  -3   0|
    | 0  -3   9  -3|
    | 0   0  -3   6|

F_uniforme    = ( 120, 150, 150, 100)    # travessia uniforme
F_aglomeracao = (  50, 300, 300,  50)    # aglomeração de pessoas no meio do vão
F_assimetrica = ( 250, 180,  60,  30)    # grupo concentrado perto de uma extremidade
```

### NÚCLEO
Decomponha `K` em LU **uma única vez** (com `scipy.linalg.lu_factor`) e reaproveite a
decomposição (`lu_solve`) para calcular o deslocamento em cada um dos três cenários:
`d_uniforme`, `d_aglomeracao`, `d_assimetrica`.

In [ ]:
# Etapa 4 — Núcleo
K = np.array([
    [8., -3., 0., 0.],
    [-3., 9., -3., 0.],
    [0., -3., 9., -3.],
    [0., 0., -3., 6.],
])
F_uniforme = np.array([120., 150., 150., 100.])
F_aglomeracao = np.array([50., 300., 300., 50.])
F_assimetrica = np.array([250., 180., 60., 30.])

# Escreva aqui o código: decomponha K em LU uma vez e resolva os 3 cenários

lu_piv = sl.lu_factor(K)
d_uniforme = sl.lu_solve(lu_piv, F_uniforme)
d_aglomeracao = sl.lu_solve(lu_piv, F_aglomeracao)
d_assimetrica = sl.lu_solve(lu_piv, F_assimetrica)

In [ ]:
# Validação — Etapa 4 Núcleo (não modifique esta célula)
import numpy as np

for nome in ["d_uniforme", "d_aglomeracao", "d_assimetrica"]:
    assert nome in dir(), f"Defina a variável {nome}"

assert np.allclose(K @ d_uniforme, F_uniforme, atol=1e-6), "d_uniforme não satisfaz K@d = F_uniforme"
assert np.allclose(K @ d_aglomeracao, F_aglomeracao, atol=1e-6), "d_aglomeracao não satisfaz K@d = F_aglomeracao"
assert np.allclose(K @ d_assimetrica, F_assimetrica, atol=1e-6), "d_assimetrica não satisfaz K@d = F_assimetrica"

print("Etapa 4 (Núcleo) OK!")
print("  Deslocamento máximo (uniforme):   ", round(float(np.max(d_uniforme)), 3), "mm")
print("  Deslocamento máximo (aglomeração):", round(float(np.max(d_aglomeracao)), 3), "mm")
print("  Deslocamento máximo (assimétrica):", round(float(np.max(d_assimetrica)), 3), "mm")


### APROFUNDAMENTO — Etapa 4 (desafio opcional)

Compare o deslocamento máximo entre os três cenários. O cenário de aglomeração (pessoas
paradas, aglomeradas no meio do vão) tende a produzir deslocamentos bem maiores que uma
travessia uniforme, mesmo com a mesma força total aplicada — pesquise brevemente por que
aglomeração e caminhada sincronizada de pedestres são consideradas casos críticos no projeto
de passarelas reais (isso já causou problemas documentados em pontes pedestres famosas). Escreva
um parágrafo curto relacionando o que você pesquisou com os números que calculou.

**Minha pesquisa e conclusão:**

Aglomeração e caminhada sincronizada são casos críticos porque concentram a carga no ponto mais flexível do vão e podem excitar a frequência natural da estrutura (ressonância), como na Millennium Bridge, em Londres, que balançou na inauguração em 2000. Nos cálculos, o deslocamento máximo foi ≈ 70,2 mm na aglomeração, contra 43,1 mm (uniforme) e 48,3 mm (assimétrica), (a carga total também é maior: 700 N contra 520 N nos outros dois cenários).

---
## Etapa 5 — Consistência de sensores redundantes

*(Conceito: matrizes de projeção)*

Por segurança, dois sensores redundantes foram instalados no mesmo ponto da passarela — ambos
deveriam, idealmente, medir o mesmo valor. A direção "consistente" (onde os dois sensores
concordam) é dada pelo vetor $v = (1, 1)^T$. As matrizes de projeção sobre essa direção e sobre o
complemento ortogonal (a "discrepância" entre os sensores) são:

$$P = \frac{v v^T}{v^T v} \qquad Q = I - P$$

### NÚCLEO
1. Construa `P5` e `Q5` a partir de `v5 = [[1.],[1.]]`.
2. Para a leitura `m5 = [[12.3],[11.7]]` (mm), calcule a parte consistente `consistente5 = P5 @ m5`
   e a parte de discrepância `discrepancia5 = Q5 @ m5`.

In [ ]:
# Etapa 5 — Núcleo
v5 = np.array([[1.], [1.]])
m5 = np.array([[12.3], [11.7]])

# Escreva aqui o código dos itens 1 e 2

P5 = (v5 @ v5.T) / (v5.T @ v5)
Q5 = np.eye(2) - P5
consistente5 = P5 @ m5
discrepancia5 = Q5 @ m5

In [ ]:
# Validação — Etapa 5 Núcleo (não modifique esta célula)
import numpy as np

assert "P5" in dir() and "Q5" in dir(), "Defina P5 e Q5"
P5_esperado = (v5 @ v5.T) / (v5.T @ v5)
Q5_esperado = np.eye(2) - P5_esperado
assert np.allclose(np.array(P5, dtype=float), P5_esperado, atol=1e-6), "P5 não confere"
assert np.allclose(np.array(Q5, dtype=float), Q5_esperado, atol=1e-6), "Q5 não confere"

assert "consistente5" in dir() and "discrepancia5" in dir(), "Defina consistente5 e discrepancia5"
assert np.allclose(np.array(consistente5, dtype=float), P5_esperado @ m5, atol=1e-6), "consistente5 não confere"
assert np.allclose(np.array(discrepancia5, dtype=float), Q5_esperado @ m5, atol=1e-6), "discrepancia5 não confere"

print("Etapa 5 (Núcleo) OK! Parte consistente:", np.array(consistente5).ravel(),
      " Discrepância:", np.array(discrepancia5).ravel())


### APROFUNDAMENTO — Etapa 5 (desafio opcional)

Um segundo par de leituras, `m5b = [[15.0],[9.5]]`, mostra uma discrepância bem maior — sinal de
possível falha ou desalinhamento em um dos sensores. Calcule `discrepancia5b = Q5 @ m5b` e a norma
`norma_discrepancia5b = np.linalg.norm(discrepancia5b)`.

Na célula de texto: compare `norma_discrepancia5b` com a norma da discrepância do NÚCLEO
(`m5`). Proponha (e justifique) um valor de limiar (threshold), em mm, acima do qual a equipe de
manutenção deveria ser acionada para inspecionar o par de sensores.

In [ ]:
# Etapa 5 — Aprofundamento
m5b = np.array([[15.0], [9.5]])

# Escreva aqui o código para calcular discrepancia5b e norma_discrepancia5b

discrepancia5b = Q5 @ m5b
norma_discrepancia5b = np.linalg.norm(discrepancia5b)
print(norma_discrepancia5b, np.linalg.norm(discrepancia5))

**Meu limiar proposto e justificativa:**

Norma da discrepância: ≈ 0,42 mm (m5, normal) contra ≈ 3,89 mm (m5b, ~9× maior). Proponho limiar de 1,0 mm: fica acima da variação normal observada (0,42 mm) e bem abaixo da falha evidente (3,89 mm), permitindo inspeção antes de a falha crescer.

In [ ]:
# Validação — Etapa 5 Aprofundamento (não modifique esta célula)
import numpy as np

if "discrepancia5b" in dir() and "norma_discrepancia5b" in dir():
    Q5_esperado = np.eye(2) - (v5 @ v5.T) / (v5.T @ v5)
    esperado_disc = Q5_esperado @ m5b
    assert np.allclose(np.array(discrepancia5b, dtype=float), esperado_disc, atol=1e-6), "discrepancia5b não confere"
    assert np.isclose(norma_discrepancia5b, np.linalg.norm(esperado_disc), atol=1e-6), "norma_discrepancia5b não confere"
    print("Etapa 5 (Aprofundamento) OK! norma_discrepancia5b ≈", round(float(norma_discrepancia5b), 4), "mm")
else:
    print("Etapa 5 (Aprofundamento) não resolvida — tudo bem, é opcional.")


---
## Etapa 6 (bônus, totalmente opcional) — Robustez numérica do modelo estrutural

*(Conceito: fator de crescimento na fatoração LU)*

Esta etapa é recomendada apenas para quem quer se aprofundar em métodos numéricos (por exemplo,
alunos pensando em TCC na área) — **não afeta a nota do projeto principal**.

À medida que um modelo estrutural é refinado (mais nós/elementos), a fatoração LU de certas
matrizes pode sofrer crescimento numérico extremo se a fatoração for feita sem pivotamento. A
matriz `m × m` abaixo (índices de 1 a `m`) é um exemplo clássico desse comportamento:

$$
A_{ij} = \begin{cases}
0 & \text{se } i < j \text{ e } j \neq m \\
1 & \text{se } i = j \text{ ou } j = m \\
-1 & \text{caso contrário}
\end{cases}
$$

1. Escreva `matriz_crescimento(m)` para montar essa matriz.
2. Calcule o fator de crescimento $\rho = \max|U_{ij}| / \max|A_{ij}|$ (via
   `scipy.linalg.lu`) para `m = 5` e `m = 10`, guardando em `rho_m5` e `rho_m10`.
3. Na célula de texto, relacione o que você observou com a importância do pivotamento parcial
   (visto no Dia 1) à medida que um modelo estrutural é refinado.

In [ ]:
# Etapa 6 — Bônus (opcional)
import scipy.linalg as sl

def matriz_crescimento(m):
    A = np.zeros((m, m))
    for i in range(m):
        for j in range(m):
            if i == j or j == m - 1:
                A[i, j] = 1.
            elif i > j:
                A[i, j] = -1.
    return A

def fator_crescimento(m):
    A = matriz_crescimento(m)
    U = sl.lu(A)[2]
    return np.max(np.abs(U)) / np.max(np.abs(A))

rho_m5 = fator_crescimento(5)
rho_m10 = fator_crescimento(10)
print(rho_m5, rho_m10)

**Minha conclusão:**

ρ = 16 (m = 5) e 512 (m = 10), ou seja, 2^(m−1): o crescimento é exponencial com o tamanho do modelo. Em modelos refinados, erros de arredondamento seriam amplificados, então a estabilidade numérica da LU deve ser sempre verificada (e o pivotamento parcial é indispensável na prática).

In [ ]:
# Validação — Etapa 6 (não modifique esta célula)
import numpy as np

if "matriz_crescimento" in dir():
    esperado_m3 = np.array([[1.,0.,1.],[-1.,1.,1.],[-1.,-1.,1.]])
    assert np.allclose(np.array(matriz_crescimento(3), dtype=float), esperado_m3), \
        "matriz_crescimento(3) não confere com o padrão esperado"
    if "rho_m5" in dir() and "rho_m10" in dir():
        assert np.isclose(rho_m5, 16.0, rtol=1e-3), "rho_m5 não confere com o valor esperado"
        assert np.isclose(rho_m10, 512.0, rtol=1e-3), "rho_m10 não confere com o valor esperado"
        print("Etapa 6 (Bônus) OK!")
    else:
        print("matriz_crescimento definida, mas rho_m5/rho_m10 ainda não — complete o item 2.")
else:
    print("Etapa 6 (Bônus) não resolvida — tudo bem, é totalmente opcional.")


---
### Entrega

Ao terminar (pelo menos o NÚCLEO de todas as 5 etapas principais), faça commit e push deste
notebook:

```bash
git add projeto_engenharia_modulo2.ipynb
git commit -m "Módulo 2: mini-projeto de monitoramento estrutural (passarela instrumentada)"
git push
```

**Critério de avaliação:** o NÚCLEO das Etapas 1 a 5 vale a nota completa do projeto. Os itens de
APROFUNDAMENTO (Etapas 1 a 5) e a Etapa 6 (bônus) somam pontos extras — resolva quantos fizer
sentido para o seu momento no curso.